# GAT-v4 | Phase 8 — Pathway Analysis

**Covers plan section 11 (Phase 10)**

### Pipeline
1. Load best checkpoint for each of the 15 folds
2. Run forward pass on each test patient → extract gene embeddings `h_gene` (4439 × 64)
3. Scatter-mean over pathway gene nodes → pathway embedding (64,)
4. Pathway dysregulation score = L2 norm of mean pathway embedding
5. Aggregate scores across folds (each patient appears in test 3× — one per seed)
6. Rank pathways by **IQR × Δ** (variability × risk-group separation)
7. Select Top-K (K=15–20), run univariate Cox per pathway (BH-corrected)
8. Target: ≥ 5 pathways at BH q < 0.05

### Memory strategy
Gene embeddings (4439 × 64 per patient) are computed on-the-fly and immediately
projected to pathway scores — never stored in full to avoid OOM.

---
| Cell | What |
|------|------|
| 1 | Imports & paths |
| 2 | Load gene index, Reactome, clinical |
| 3 | Build pathway → node index map |
| 4 | Extract pathway scores + risk per fold |
| 5 | Aggregate across folds per patient |
| 6 | Rank pathways (IQR × Δ) |
| 7 | Univariate Cox + BH correction |
| 8 | Top-K selection + summary table |
| 9 | Save outputs + audit YAML |

---
## Cell 1 — Imports & Paths

In [1]:
import sys, json, yaml, hashlib, time
from pathlib import Path
from datetime import datetime
from collections import defaultdict

import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv
from lifelines import CoxPHFitter
from tqdm.auto import tqdm

# ── PATHS ─────────────────────────────────────────────────────────────────────
PROJECT_ROOT = Path(r"C:\Users\jiyad\Desktop\PROJECT\KIRC\script")
KIRC_ROOT   = PROJECT_ROOT / "KIRC"
DATA_PROC    = KIRC_ROOT / "data" / "processed"
DATA_SPLITS  = KIRC_ROOT / "data" / "splits"
CKPT_DIR     = KIRC_ROOT / "results" / "checkpointsibsLOG"
LOG_DIR      = KIRC_ROOT / "results" / "logs"
TABLE_DIR    = KIRC_ROOT / "results" / "tables"
AUDIT_DIR    = KIRC_ROOT / "audit"
# ─────────────────────────────────────────────────────────────────────────────

TABLE_DIR.mkdir(parents=True, exist_ok=True)

if str(KIRC_ROOT) not in sys.path:
    sys.path.insert(0, str(KIRC_ROOT))
from src.utils.logger import get_logger
from src.config import GAT_IN_DIM, GAT_HIDDEN, GAT_HEADS, GAT_MLP_HIDDEN, GAT_CLINICAL_DIM, GAT_DROPOUT

log    = get_logger('phase8_pathway')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {DEVICE}")


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\jiyad\Desktop\PROJECT\KIRC\script\.venv\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\jiyad\Desktop\PROJECT\KIRC\script\.venv\Lib\site-packages\traitlets\config\application.py", line 1082, in launch_instance
    app.start()
  File "C:\Users\jiyad\Desktop\PROJECT\KIRC\script\.venv\Lib\site-packages\ipykernel\kernelapp.py", line 807, in start

Device : cuda


---
## Cell 2 — Load Gene Index, Reactome, Clinical

In [2]:
# ── Gene index (PPI node ordering) ───────────────────────────────────────────
gene_idx_df = pd.read_csv(DATA_PROC / 'ppi_gene_index.csv')
ppi_genes   = gene_idx_df['gene_symbol'].tolist()
gene_to_idx = {g: i for i, g in enumerate(ppi_genes)}
n_nodes     = len(ppi_genes)
print(f"PPI nodes : {n_nodes:,}")

# ── Reactome filtered pathways ────────────────────────────────────────────────
reactome_df = pd.read_csv(DATA_PROC / 'reactome_filtered.csv')
print(f"Pathways  : {len(reactome_df):,}")

# Build pathway_id → set of gene symbols
pathway_genes = {}
pathway_names = {}
for _, row in reactome_df.iterrows():
    pid   = row['reactome_id']
    genes = set(row['gene_symbols'].split('|'))
    pathway_genes[pid] = genes
    pathway_names[pid] = row['pathway_name']

# ── Clinical ──────────────────────────────────────────────────────────────────
clin = pd.read_csv(DATA_PROC / 'clinical_clean.csv').set_index('patient_id')
print(f"Clinical  : {clin.shape}")

# ── CV splits + sample alignment ─────────────────────────────────────────────
with open(DATA_SPLITS / 'cv_folds_repeated_5x3.json') as f:
    cv_splits = json.load(f)

align       = pd.read_csv(DATA_PROC / 'sample_alignment.csv')
align_tumor = align[(align['tissue_type']=='tumor') & align['include']]
pat_to_sids = align_tumor.groupby('patient_id')['sample_id'].apply(list).to_dict()
sid_to_pat  = dict(zip(align_tumor['sample_id'], align_tumor['patient_id']))

print(f"CV folds  : {len(cv_splits)}")
print("\nCell 2 done.")

PPI nodes : 4,414
Pathways  : 807
Clinical  : (507, 5)
CV folds  : 15

Cell 2 done.


---
## Cell 3 — Build Pathway → Node Index Map

Maps each Reactome pathway to the integer node indices of its genes in the PPI graph.
Pathways with fewer than 3 nodes in the PPI are excluded (insufficient signal).

In [3]:
MIN_PPI_NODES = 3   # minimum PPI nodes per pathway

pathway_node_idx = {}   # pathway_id → np.array of node indices
excluded = []

for pid, genes in pathway_genes.items():
    node_indices = np.array(
        [gene_to_idx[g] for g in genes if g in gene_to_idx],
        dtype=np.int32
    )
    if len(node_indices) >= MIN_PPI_NODES:
        pathway_node_idx[pid] = node_indices
    else:
        excluded.append(pid)

pathway_ids = list(pathway_node_idx.keys())
n_pathways  = len(pathway_ids)

print(f"Pathways with >= {MIN_PPI_NODES} PPI nodes : {n_pathways:,}")
print(f"Excluded (< {MIN_PPI_NODES} PPI nodes)    : {len(excluded):,}")

# Distribution of pathway PPI node counts
sizes = np.array([len(pathway_node_idx[pid]) for pid in pathway_ids])
print(f"\nPPI nodes per pathway:")
print(f"  Min    : {sizes.min()}")
print(f"  Median : {np.median(sizes):.0f}")
print(f"  Mean   : {sizes.mean():.1f}")
print(f"  Max    : {sizes.max()}")
print("\nCell 3 done.")

Pathways with >= 3 PPI nodes : 805
Excluded (< 3 PPI nodes)    : 2

PPI nodes per pathway:
  Min    : 3
  Median : 11
  Mean   : 24.5
  Max    : 418

Cell 3 done.


---
## Cell 4 — Extract Pathway Scores + Risk Per Fold

For each fold:
1. Load best checkpoint → rebuild GATv4
2. For each test patient, run forward pass → get `h_gene` (n_nodes × 64)
3. For each pathway, compute `mean(h_gene[pathway_nodes])` → L2 norm → pathway score
4. Also record risk score and survival labels
5. Free GPU memory before next fold

**Runtime: ~10–20 min total** (~1 min per fold)

In [4]:
# Add this cell BEFORE the fold loop (after Cell 3)
# Convert all pathway node index arrays to Python lists once
pathway_node_idx_lists = {
    pid: idx_arr.tolist()
    for pid, idx_arr in pathway_node_idx.items()
}
print(f"Pre-converted {len(pathway_node_idx_lists)} pathway index arrays to Python lists.")
# Re-define GATv4 (same architecture as Phase 7)
class GATv4(nn.Module):
    def __init__(self, in_dim=GAT_IN_DIM, hidden=GAT_HIDDEN, heads=GAT_HEADS,
                 mlp_hidden=GAT_MLP_HIDDEN, clinical_dim=GAT_CLINICAL_DIM,
                 dropout=GAT_DROPOUT):
        super().__init__()
        self.dropout = dropout
        hd = hidden // heads
        self.gat1     = GATConv(in_dim,  hd, heads=heads, concat=True,
                                dropout=dropout, edge_dim=1, add_self_loops=True)
        self.norm1    = nn.LayerNorm(hidden)
        self.gat2     = GATConv(hidden, hd, heads=heads, concat=True,
                                dropout=dropout, edge_dim=1, add_self_loops=True)
        self.norm2    = nn.LayerNorm(hidden)
        self.residual = nn.Linear(in_dim, hidden, bias=False)
        self.risk_head = nn.Sequential(
            nn.Linear(hidden + clinical_dim, mlp_hidden), nn.ELU(),
            nn.Dropout(dropout), nn.Linear(mlp_hidden, 1)
        )

    def forward(self, x, edge_index, edge_weight, clinical):
        ew  = edge_weight.unsqueeze(-1)
        h   = F.elu(self.norm1(self.gat1(x, edge_index, edge_attr=ew)))
        h   = F.dropout(h, self.dropout, self.training)
        h2  = self.norm2(self.gat2(h, edge_index, edge_attr=ew))
        hg  = F.elu(h2 + self.residual(x))
        hg  = F.dropout(hg, self.dropout, self.training)
        zp  = hg.mean(dim=0)
        risk = self.risk_head(torch.cat([zp, clinical], dim=-1)).squeeze()
        return risk, hg


# Collector: patient_id → list of (pathway_scores, risk_score) across folds
# pathway_scores shape per entry: (n_pathways,)
patient_pathway_scores = defaultdict(list)  # patient_id → list of np arrays
patient_risk_scores    = defaultdict(list)  # patient_id → list of floats
patient_survival       = {}                 # patient_id → (os_months, event)

t_total = time.time()

for fold_key, fold_def in tqdm(cv_splits.items(), desc='Extracting embeddings'):
    ckpt_path = CKPT_DIR / f"{fold_key}_best_model.pt"
    if not ckpt_path.exists():
        print(f"  WARNING: checkpoint missing for {fold_key} — skipping")
        continue

    # Load checkpoint
    ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    model = GATv4().to(DEVICE)
    model.load_state_dict(ckpt['model_state'])
    model.eval()

    # Load fold tensors
    fold_dir = DATA_SPLITS / fold_key
    feat = torch.tensor(np.load(fold_dir / 'feature_tensor.npy'),  dtype=torch.float32)
    clin_t = torch.tensor(np.load(fold_dir / 'clinical_tensor.npy'), dtype=torch.float32)
    surv_t = torch.tensor(np.load(fold_dir / 'survival_labels.npy'), dtype=torch.float32)
    ei   = torch.tensor(np.load(fold_dir / 'edge_index.npy'),  dtype=torch.long).to(DEVICE)
    ew   = torch.tensor(np.load(fold_dir / 'edge_weight.npy'), dtype=torch.float32).to(DEVICE)

    with open(fold_dir / 'sample_ids.json') as f:
        fold_sids = json.load(f)
    with open(fold_dir / 'split_indices.json') as f:
        split_idx = json.load(f)

    # Process test patients only
    test_indices = split_idx['test']

    with torch.no_grad():
        for i in test_indices:
            sid = fold_sids[i]
            pat = sid_to_pat.get(sid)
            if pat is None:
                continue

            x = feat[i].to(DEVICE)     # (n_nodes, 3)
            c = clin_t[i].to(DEVICE)   # (3,)

            risk, h_gene = model(x, ei, ew, c)
            # h_gene: (n_nodes, 64) on GPU

            # Compute pathway scores (on CPU to free GPU memory per patient)
            hg_cpu = torch.tensor(h_gene.tolist(), dtype=torch.float32)  # bridge-free

            path_scores = np.zeros(n_pathways, dtype=np.float32)
            for p_idx, pid in enumerate(pathway_ids):
                # Convert int32 numpy array → Python list to avoid numpy bridge
                node_idx_list = pathway_node_idx[pid].tolist()
                path_emb = hg_cpu[node_idx_list].mean(dim=0)   # (64,)
                path_scores[p_idx] = float(path_emb.norm().item())

            patient_pathway_scores[pat].append(path_scores)
            patient_risk_scores[pat].append(risk.item())

            # Store survival (same across folds)
            if pat not in patient_survival:
                os_m  = float(surv_t[i, 0].item())
                event = float(surv_t[i, 1].item())
                patient_survival[pat] = (os_m, event)

    del model, feat, clin_t, surv_t, ei, ew
    torch.cuda.empty_cache()

print(f"\nDone in {(time.time()-t_total)/60:.1f} min")
print(f"Patients with pathway scores : {len(patient_pathway_scores):,}")
print(f"Expected: ~{len(cv_splits)//5 * 104} (15 folds, each patient in test 3×)")
log.info(f"Embedding extraction complete: {len(patient_pathway_scores)} patients")
print("\nCell 4 done.")

Pre-converted 805 pathway index arrays to Python lists.


Extracting embeddings: 100%|██████████| 15/15 [05:05<00:00, 20.40s/it]
2026-07-13 22:30:21 | INFO     | phase8_pathway | Embedding extraction complete: 491 patients



Done in 5.1 min
Patients with pathway scores : 491
Expected: ~312 (15 folds, each patient in test 3×)

Cell 4 done.


---
## Cell 5 — Aggregate Across Folds Per Patient

Each patient appears in the test set 3× (once per seed).
Average pathway scores and risk scores across those 3 fold appearances.

In [5]:
patients_list   = sorted(patient_pathway_scores.keys())
n_patients      = len(patients_list)

# Shape: (n_patients, n_pathways)
score_matrix = np.zeros((n_patients, n_pathways), dtype=np.float32)
risk_scores  = np.zeros(n_patients, dtype=np.float32)
os_months    = np.zeros(n_patients, dtype=np.float64)
events       = np.zeros(n_patients, dtype=np.float64)

for i, pat in enumerate(patients_list):
    fold_scores = patient_pathway_scores[pat]   # list of (n_pathways,) arrays
    score_matrix[i] = np.mean(fold_scores, axis=0)  # average across folds
    risk_scores[i]  = np.mean(patient_risk_scores[pat])
    os_months[i], events[i] = patient_survival[pat]

print(f"Score matrix : {score_matrix.shape}  (patients × pathways)")
print(f"Risk scores  : {risk_scores.shape}")
print(f"Events       : {int(events.sum())} / {n_patients} patients")
print(f"\nScore value range : [{score_matrix.min():.3f}, {score_matrix.max():.3f}]")
print(f"Score mean        : {score_matrix.mean():.3f}")

# Mean appearances per patient (should be ~3 for 3 seeds)
appearances = [len(v) for v in patient_pathway_scores.values()]
print(f"\nFold appearances per patient: mean={np.mean(appearances):.1f}  "
      f"(expected 3.0 for 3 seeds)")
print("\nCell 5 done.")

Score matrix : (491, 805)  (patients × pathways)
Risk scores  : (491,)
Events       : 160 / 491 patients

Score value range : [2.054, 14.208]
Score mean        : 3.912

Fold appearances per patient: mean=3.1  (expected 3.0 for 3 seeds)

Cell 5 done.


---
## Cell 6 — Rank Pathways (IQR × Δ)

**IQR** = pathway score 75th–25th percentile across all patients (measures variability)

**Δ** = |mean score high-risk − mean score low-risk| where the split is by median risk score

High IQR × Δ = pathway that varies a lot AND separates risk groups.

In [6]:
# Split patients into high/low risk by median risk score
risk_median  = np.median(risk_scores)
high_risk_mask = risk_scores >= risk_median
low_risk_mask  = ~high_risk_mask

print(f"Risk median : {risk_median:.4f}")
print(f"High risk   : {high_risk_mask.sum()} patients")
print(f"Low risk    : {low_risk_mask.sum()} patients")
print()

pathway_stats = []
for p_idx, pid in enumerate(pathway_ids):
    scores = score_matrix[:, p_idx]

    q75, q25 = np.percentile(scores, 75), np.percentile(scores, 25)
    iqr = q75 - q25

    mean_high = scores[high_risk_mask].mean()
    mean_low  = scores[low_risk_mask].mean()
    delta     = abs(mean_high - mean_low)

    pathway_stats.append({
        'reactome_id'   : pid,
        'pathway_name'  : pathway_names.get(pid, pid),
        'n_ppi_nodes'   : len(pathway_node_idx[pid]),
        'iqr'           : round(float(iqr), 5),
        'delta'         : round(float(delta), 5),
        'iqr_x_delta'   : round(float(iqr * delta), 6),
        'mean_high_risk': round(float(mean_high), 5),
        'mean_low_risk' : round(float(mean_low), 5),
    })

stats_df = pd.DataFrame(pathway_stats).sort_values('iqr_x_delta', ascending=False)

print("Top 20 pathways by IQR × Δ:")
print(stats_df[['pathway_name','n_ppi_nodes','iqr','delta','iqr_x_delta']]
      .head(20).to_string(index=False))
print("\nCell 6 done.")

Risk median : 0.0482
High risk   : 246 patients
Low risk    : 245 patients

Top 20 pathways by IQR × Δ:
                                                                 pathway_name  n_ppi_nodes     iqr   delta  iqr_x_delta
                                                 Regulation of CDH11 function            3 2.25146 0.40589     0.913841
 Negative regulation of activity of TFAP2 (AP-2) family transcription factors            5 1.73646 0.48678     0.845267
                      Regulation of cholesterol biosynthesis by SREBP (SREBF)            6 1.56836 0.50125     0.786140
                                          Transcriptional Regulation by VENTX            7 1.59455 0.47741     0.761251
Regulation of MITF-M-dependent genes involved in cell cycle and proliferation            5 1.42350 0.51688     0.735774
                                       Specification of primordial germ cells            6 1.36892 0.53336     0.730133
                                                      RU

---
## Cell 7 — Univariate Cox + BH Correction

For each of the top-50 pathways by IQR × Δ:
- Fit univariate Cox (pathway score → survival)
- Extract HR, p-value, 95% CI
- BH-correct p-values

**Target: ≥ 5 pathways at BH q < 0.05**

In [7]:
import warnings
warnings.filterwarnings('ignore')

TOP_K_FOR_COX = 50   # test top-50 by IQR×Δ, then BH-correct

top50_ids = stats_df['reactome_id'].head(TOP_K_FOR_COX).tolist()
cox_results = []

surv_df_base = pd.DataFrame({
    'os_months': os_months,
    'event'    : events
})

for pid in tqdm(top50_ids, desc='Univariate Cox'):
    p_idx  = pathway_ids.index(pid)
    scores = score_matrix[:, p_idx]

    # Standardise score for stable Cox fitting
    scores_std = (scores - scores.mean()) / (scores.std() + 1e-8)

    surv_df = surv_df_base.copy()
    surv_df['pathway_score'] = scores_std
    surv_df = surv_df[surv_df['os_months'] > 0].copy()

    try:
        cph = CoxPHFitter(penalizer=0.1)
        cph.fit(surv_df, duration_col='os_months', event_col='event',
                formula='pathway_score', show_progress=False)
        summary = cph.summary

        hr     = float(summary.loc['pathway_score', 'exp(coef)'])
        hr_lo  = float(summary.loc['pathway_score', 'exp(coef) lower 95%'])
        hr_hi  = float(summary.loc['pathway_score', 'exp(coef) upper 95%'])
        pval   = float(summary.loc['pathway_score', 'p'])

        cox_results.append({
            'reactome_id'  : pid,
            'pathway_name' : pathway_names.get(pid, pid),
            'n_ppi_nodes'  : len(pathway_node_idx[pid]),
            'iqr_x_delta'  : stats_df[stats_df['reactome_id']==pid]['iqr_x_delta'].values[0],
            'hr'           : round(hr,    4),
            'hr_lo_95'     : round(hr_lo, 4),
            'hr_hi_95'     : round(hr_hi, 4),
            'pval'         : pval,
        })
    except Exception as e:
        cox_results.append({
            'reactome_id': pid, 'pathway_name': pathway_names.get(pid,''),
            'n_ppi_nodes': len(pathway_node_idx[pid]),
            'iqr_x_delta': 0.0, 'hr': np.nan,
            'hr_lo_95': np.nan, 'hr_hi_95': np.nan, 'pval': 1.0
        })

cox_df = pd.DataFrame(cox_results)

# BH correction
pvals = cox_df['pval'].fillna(1.0).values
_, padj, _, _ = multipletests(pvals, method='fdr_bh')
cox_df['padj'] = padj
cox_df = cox_df.sort_values('padj')

n_sig = (cox_df['padj'] < 0.05).sum()
print(f"\nPathways tested (top-50 by IQR×Δ) : {len(cox_df)}")
print(f"Significant at BH q<0.05           : {n_sig}  (target ≥ 5)")
print()
print("Top 15 by BH-adjusted p-value:")
display_cols = ['pathway_name','n_ppi_nodes','hr','hr_lo_95','hr_hi_95','pval','padj']
print(cox_df[display_cols].head(15).to_string(index=False))

if n_sig < 5:
    print(f"\nWARNING: Only {n_sig} pathways significant (target ≥ 5).")
    print("  This triggers the plan's fallback: shift narrative focus to biomarker panel.")
    print("  Document this honestly in manuscript Discussion.")

log.info(f"Pathway Cox: {n_sig}/{len(cox_df)} significant at BH q<0.05")
print("\nCell 7 done.")

Univariate Cox: 100%|██████████| 50/50 [00:05<00:00,  8.91it/s]
2026-07-13 22:31:07 | INFO     | phase8_pathway | Pathway Cox: 31/50 significant at BH q<0.05



Pathways tested (top-50 by IQR×Δ) : 50
Significant at BH q<0.05           : 31  (target ≥ 5)

Top 15 by BH-adjusted p-value:
                                                                                          pathway_name  n_ppi_nodes     hr  hr_lo_95  hr_hi_95         pval     padj
                                               Regulation of cholesterol biosynthesis by SREBP (SREBF)            6 0.6821    0.5937    0.7835 6.389109e-08 0.000003
                                                                                           SUMOylation           32 1.3878    1.2106    1.5910 2.571349e-06 0.000064
                                                                         Citric acid cycle (TCA cycle)            6 0.7160    0.6180    0.8295 8.582118e-06 0.000143
                         Regulation of MITF-M-dependent genes involved in cell cycle and proliferation            5 1.3386    1.1694    1.5323 2.333768e-05 0.000292
                                                 

---
## Cell 8 — Top-K Selection + Summary Table

Select K=15–20 pathways for reporting based on BH-corrected significance + IQR×Δ ranking.

In [8]:
TOP_K = 20

# Primary selection: BH q<0.05, sorted by padj
sig_pathways = cox_df[cox_df['padj'] < 0.05].copy()

# If fewer than TOP_K significant, fill with next-best by padj
if len(sig_pathways) < TOP_K:
    fill_n   = TOP_K - len(sig_pathways)
    not_sig  = cox_df[cox_df['padj'] >= 0.05].head(fill_n)
    top_pathways = pd.concat([sig_pathways, not_sig], ignore_index=True)
    print(f"Only {len(sig_pathways)} significant — filling to {TOP_K} with next-best padj.")
else:
    top_pathways = sig_pathways.head(TOP_K)

print(f"Top-{TOP_K} pathways selected:")
print(f"  Significant (q<0.05): {(top_pathways['padj']<0.05).sum()}")
print(f"  Not significant     : {(top_pathways['padj']>=0.05).sum()}")
print()
print(top_pathways[['pathway_name','hr','hr_lo_95','hr_hi_95','pval','padj']]
      .to_string(index=False))

# Add pathway scores for top pathways to score matrix
top_pathway_ids = top_pathways['reactome_id'].tolist()

# Build per-patient top-pathway score DataFrame for downstream use
top_scores_dict = {'patient_id': patients_list}
top_scores_dict['risk_score'] = risk_scores.tolist()
top_scores_dict['os_months']  = os_months.tolist()
top_scores_dict['event']      = events.tolist()
for pid in top_pathway_ids:
    p_idx = pathway_ids.index(pid)
    top_scores_dict[pid] = score_matrix[:, p_idx].tolist()

top_scores_df = pd.DataFrame(top_scores_dict)
print(f"\nTop pathway score matrix : {top_scores_df.shape}")
print("\nCell 8 done.")

Top-20 pathways selected:
  Significant (q<0.05): 20
  Not significant     : 0

                                                                                          pathway_name     hr  hr_lo_95  hr_hi_95         pval     padj
                                               Regulation of cholesterol biosynthesis by SREBP (SREBF) 0.6821    0.5937    0.7835 6.389109e-08 0.000003
                                                                                           SUMOylation 1.3878    1.2106    1.5910 2.571349e-06 0.000064
                                                                         Citric acid cycle (TCA cycle) 0.7160    0.6180    0.8295 8.582118e-06 0.000143
                         Regulation of MITF-M-dependent genes involved in cell cycle and proliferation 1.3386    1.1694    1.5323 2.333768e-05 0.000292
                                                                   Transcriptional regulation by RUNX3 1.3126    1.1424    1.5082 1.234018e-04 0.000939
        

---
## Cell 9 — Save Outputs + Audit YAML

In [10]:
def file_md5(path):
    h = hashlib.md5()
    with open(path,'rb') as f:
        while d := f.read(8192): h.update(d)
    return h.hexdigest()

# Full pathway stats (all tested)
full_path = TABLE_DIR / 'pathway_cox_results.csv'
cox_df.to_csv(full_path, index=False)
print(f"Full Cox results -> {full_path}")

# Top-K pathway score matrix per patient
scores_path = TABLE_DIR / 'top_pathway_scores.csv'
top_scores_df.to_csv(scores_path, index=False)
print(f"Top pathway scores -> {scores_path}")

# Full score matrix (all pathways × all patients) — for biomarker phase
all_scores_df = pd.DataFrame(
    score_matrix,
    index   = patients_list,
    columns = pathway_ids
)
all_scores_df.index.name = 'patient_id'
all_scores_path = TABLE_DIR / 'all_pathway_scores.csv'
all_scores_df.to_csv(all_scores_path)
print(f"All pathway scores -> {all_scores_path}")

# Risk scores per patient
risk_df = pd.DataFrame({
    'patient_id': patients_list,
    'risk_score': risk_scores,
    'os_months' : os_months,
    'event'     : events
})
risk_path = TABLE_DIR / 'patient_risk_scores.csv'
risk_df.to_csv(risk_path, index=False)
print(f"Risk scores -> {risk_path}")

# Audit YAML
audit = {
    'phase'    : 'Phase 8 — Pathway Analysis',
    'timestamp': datetime.now().isoformat(),

    'method': {
        'embedding_source' : 'GATv4 h_gene (post-GAT2 + residual), 15 fold checkpoints',
        'pathway_score'    : 'L2 norm of scatter_mean(h_gene[pathway_nodes])',
        'aggregation'      : 'Mean across 3 seed appearances per patient',
        'ranking_criterion': 'IQR x Delta (pathway score IQR x |high_risk_mean - low_risk_mean|)',
        'cox_penalizer'    : 0.1,
        'correction'       : 'BH FDR',
    },

    'results': {
        'n_pathways_tested'     : len(cox_df),
        'n_significant_q05'     : int(n_sig),
        'target_gate'           : '>= 5 pathways at BH q<0.05',
        'gate_passed'           : bool(n_sig >= 5),
        'top_k_selected'        : TOP_K,
        'top_pathway_ids'       : top_pathway_ids[:5],
        'top_pathway_names'     : top_pathways['pathway_name'].head(5).tolist(),
        'top_hr_range'          : [round(float(top_pathways['hr'].min()),3),
                                   round(float(top_pathways['hr'].max()),3)],
    },

    'output_checksums': {
        'pathway_cox_results.csv': file_md5(full_path),
        'patient_risk_scores.csv': file_md5(risk_path),
    },

    'next_phase': 'Phase 9 — Biomarker Discovery (Integrated Gradients)',
    'status'    : 'COMPLETE',
}

audit_path = AUDIT_DIR / 'phase8_pathway_analysis.yaml'
with open(audit_path, 'w', encoding='utf-8') as f:
    yaml.dump(audit, f, default_flow_style=False, sort_keys=False)
print(f"Audit YAML -> {audit_path}")

print()
print("=" * 55)
print("PHASE 8 COMPLETE")
print("=" * 55)
print(f"  Pathways tested      : {len(cox_df)}")
print(f"  Significant (q<0.05) : {n_sig}  {'PASS' if n_sig>=5 else 'FAIL — see fallback'}")
print(f"  Top-{TOP_K} selected")
print()
print("Next: 09_biomarker_ig.ipynb — Integrated Gradients biomarker panel")

Full Cox results -> C:\Users\jiyad\Desktop\PROJECT\KIRC\script\KIRC\results\tables\pathway_cox_results.csv
Top pathway scores -> C:\Users\jiyad\Desktop\PROJECT\KIRC\script\KIRC\results\tables\top_pathway_scores.csv
All pathway scores -> C:\Users\jiyad\Desktop\PROJECT\KIRC\script\KIRC\results\tables\all_pathway_scores.csv
Risk scores -> C:\Users\jiyad\Desktop\PROJECT\KIRC\script\KIRC\results\tables\patient_risk_scores.csv
Audit YAML -> C:\Users\jiyad\Desktop\PROJECT\KIRC\script\KIRC\audit\phase8_pathway_analysis.yaml

PHASE 8 COMPLETE
  Pathways tested      : 50
  Significant (q<0.05) : 31  PASS
  Top-20 selected

Next: 09_biomarker_ig.ipynb — Integrated Gradients biomarker panel


---
## Cell 10 — Final Summary Table (Pathway ID, Name, Dysregulation Score, Hazard Ratio)

In [11]:
# ── Final summary table: pathway id, name, dysregulation score, hazard ratio ─
# 'Dysregulation score' = IQR x Delta (the same high/low-risk separation metric
# used upstream to rank pathways before Cox testing).
summary_table = top_pathways[['reactome_id', 'pathway_name', 'iqr_x_delta', 'hr']].copy()
summary_table.columns = ['pathway_id', 'pathway_name', 'dysregulation_score', 'hazard_ratio']
summary_table = summary_table.sort_values('dysregulation_score', ascending=False).reset_index(drop=True)

summary_path = TABLE_DIR / 'pathway_summary_table.csv'
summary_table.to_csv(summary_path, index=False)
print(f"Pathway summary table -> {summary_path}")
print(f"Rows: {len(summary_table)}")
print()
print(summary_table.to_string(index=False))

summary_table


Pathway summary table -> C:\Users\jiyad\Desktop\PROJECT\KIRC\script\KIRC\results\tables\pathway_summary_table.csv
Rows: 20

   pathway_id                                                                                           pathway_name  dysregulation_score  hazard_ratio
R-HSA-1655829                                                Regulation of cholesterol biosynthesis by SREBP (SREBF)             0.786140        0.6821
R-HSA-9825892                          Regulation of MITF-M-dependent genes involved in cell cycle and proliferation             0.735774        1.3386
R-HSA-9827857                                                                 Specification of primordial germ cells             0.730133        1.2884
R-HSA-8951936                                                                                RUNX3 regulates p14-ARF             0.729766        1.2620
  R-HSA-71403                                                                          Citric acid cycle (TCA cycle)

,pathway_id,pathway_name,dysregulation_score,hazard_ratio
0,R-HSA-1655829,Regulation of cholesterol biosynthesis by SREB...,0.786140,0.6821
1,R-HSA-9825892,Regulation of MITF-M-dependent genes involved ...,0.735774,1.3386
2,R-HSA-9827857,Specification of primordial germ cells,0.730133,1.2884
3,R-HSA-8951936,RUNX3 regulates p14-ARF,0.729766,1.2620
4,R-HSA-71403,Citric acid cycle (TCA cycle),0.619356,0.7160
5,R-HSA-8941858,Regulation of RUNX3 expression and activity,0.607809,1.2560
6,R-HSA-933542,TRAF6 mediated NF-kB activation,0.604506,1.2454
7,R-HSA-8878159,Transcriptional regulation by RUNX3,0.588080,1.3126
8,R-HSA-162587,HIV Life Cycle,0.538063,1.2552
9,R-HSA-8857538,PTK6 promotes HIF1A stabilization,0.520292,1.2417
